In [8]:
from typing import Any
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np
import seaborn as sns
from plot_utils import cblind_cmap
from typing import Callable, Literal
import warnings


from tqdm.notebook import tqdm


plt.style.use("style.mplstyle")

PHASE_ORDERS = ["liquid", "ice", "mixed", "drizzle", "liq_driz", "rain", "snow"]
MODELS = ["cnn", "cnn_dropout", "mlp_balanced", "rf_balanced"]

In [2]:
def load_pred(filepath: str) -> pd.DataFrame:
    df = pd.read_parquet(filepath)
    df = df.reset_index()
    df["height"] = np.round(df["height"], 2)
    df["height_bin"] = pd.cut(df["height"], bins=np.arange(0, 12, 0.5), labels=np.arange(0.5, 12, 0.5))
    df = df.set_index(["time", "height"])
    return df

def load_counts_and_ratios(filepath: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    counts = pd.read_parquet(filepath)
    counts = counts.loc[:, "cloud_phase", :].reset_index() # type: ignore
    counts["height"] = np.round(counts["height"], 2)
    counts = counts.pivot(index="height", columns="phase", values="count").fillna(0).astype(int)
    counts = counts[PHASE_ORDERS]
    ratios = counts.apply(lambda row: row / row.sum(), axis=1)
    return counts, ratios

In [3]:
df = load_pred("./data/parallel_nsa_cloudy_predictions.parquet")
df

cloud_phase cnn_dropout     cnn rf_balanced  \
time                height                                               
2021-01-01 00:57:00 0.16           ice         ice     ice      liquid   
                    0.19           ice         ice     ice      liquid   
                    0.22           ice         ice     ice         ice   
                    0.25           ice         ice     ice      liquid   
2021-01-01 00:57:30 0.16           ice         ice     ice         ice   
...                                ...         ...     ...         ...   
2021-12-31 23:59:30 2.38           ice      liquid  liquid         ice   
                    2.41           ice      liquid  liquid         ice   
                    2.44           ice      liquid  liquid         ice   
                    2.47           ice      liquid  liquid         ice   
                    2.50           ice      liquid  liquid         ice   

                           rf_imbalanced mlp_balanced mlp_imbalanced  \
time                height                                             
2021-01-01 00:57:00 0.16             ice       liquid            ice   
                    0.19             ice          ice            ice   
                    0.22             ice          ice            ice   
                    0.25             ice          ice            ice   
2021-01-01 00:57:30 0.16             ice       liquid            ice   
...                                  ...          ...            ...   
2021-12-31 23:59:30 2.38             ice          ice            ice   
                    2.41             ice          ice            ice   
                    2.44             ice          ice            ice   
                    2.47             ice          ice            ice   
                    2.50             ice          ice            ice   

                            cnn_dropout_confidence  cnn_confidence  \
time                height                                           
2021-01-01 00:57:00 0.16                  0.864284        0.793064   
                    0.19                  0.920299        0.910763   
                    0.22                  0.859795        0.926132   
                    0.25                  0.812101        0.895370   
2021-01-01 00:57:30 0.16                  0.909938        0.861275   
...                                            ...             ...   
2021-12-31 23:59:30 2.38                  0.000000        0.000000   
                    2.41                  0.000000        0.000000   
                    2.44                  0.000000        0.000000   
                    2.47                  0.000000        0.000000   
                    2.50                  0.000000        0.000000   

                            rf_balanced_confidence  ...  mlp_imbalanced_mpl_b  \
time                height                          ...                         
2021-01-01 00:57:00 0.16                      0.55  ...                liquid   
                    0.19                      0.52  ...                   ice   
                    0.22                      0.55  ...                   ice   
                    0.25                      0.59  ...                   ice   
2021-01-01 00:57:30 0.16                      0.51  ...                   ice   
...                                            ...  ...                   ...   
2021-12-31 23:59:30 2.38                      0.60  ...                   ice   
                    2.41                      0.62  ...                   ice   
                    2.44                      0.73  ...                   ice   
                    2.47                      0.72  ...                   ice   
                    2.50                      0.49  ...                   ice   

                            mlp_imbalanced_mpl_ldr  mlp_imbalanced_mwr  \
time                height                                               
2021-01-01 00:

In [4]:
anx = load_pred("./data/parallel_anx_cloudy_predictions.parquet")
anx

cloud_phase cnn_dropout       cnn rf_balanced  \
time                height                                                 
2020-02-11 17:59:30 0.16      liq_driz         ice  liq_driz    liq_driz   
                    0.19       drizzle         ice  liq_driz     drizzle   
                    0.22       drizzle         ice   drizzle     drizzle   
                    0.25       drizzle         ice   drizzle     drizzle   
                    0.28       drizzle         ice   drizzle     drizzle   
...                                ...         ...       ...         ...   
2020-05-31 19:10:30 7.87           ice         ice       ice         ice   
                    7.90           ice         ice       ice         ice   
                    7.93           ice         ice       ice         ice   
                    7.96           ice         ice       ice         ice   
                    7.99           ice         ice       ice         ice   

                           rf_imbalanced mlp_balanced mlp_imbalanced  \
time                height                                             
2020-02-11 17:59:30 0.16        liq_driz      drizzle        drizzle   
                    0.19         drizzle      drizzle        drizzle   
                    0.22         drizzle      drizzle        drizzle   
                    0.25         drizzle      drizzle        drizzle   
                    0.28         drizzle      drizzle        drizzle   
...                                  ...          ...            ...   
2020-05-31 19:10:30 7.87             ice          ice            ice   
                    7.90             ice          ice            ice   
                    7.93             ice          ice            ice   
                    7.96             ice          ice            ice   
                    7.99             ice          ice            ice   

                            cnn_dropout_confidence  cnn_confidence  \
time                height                                           
2020-02-11 17:59:30 0.16                  0.467135        0.817501   
                    0.19                  0.578780        0.813098   
                    0.22                  0.764188        0.735172   
                    0.25                  0.981559        0.835488   
                    0.28                  0.993217        0.777331   
...                                            ...             ...   
2020-05-31 19:10:30 7.87                  0.999995        0.999997   
                    7.90                  0.999996        0.999999   
                    7.93                  0.999997        0.999999   
                    7.96                  0.999990        0.999999   
                    7.99                  0.998544        0.999998   

                            rf_balanced_confidence  ...  mlp_imbalanced_mpl_b  \
time                height                          ...                         
2020-02-11 17:59:30 0.16                      0.53  ...               drizzle   
                    0.19                      0.69  ...               drizzle   
                    0.22                      0.73  ...                liquid   
                    0.25                      0.72  ...               drizzle   
                    0.28                      0.70  ...                liquid   
...                                            ...  ...                   ...   
2020-05-31 19:10:30 7.87                      0.85  ...                   ice   
                    7.90                      0.80  ...                   ice   
                    7.93                      0.67  ...                   ice   
                    7.96                      0.78  ...                   ice   
                    7.99                      0.81  ...                   ice   

                            mlp_imbalanced_mpl_ldr  mlp_imbalanced_mwr  \
time                height                                    

In [5]:
COUNTS_DF, RATIOS_DF = load_counts_and_ratios("./data/nsa_phase_counts.parquet")
COUNTS_DF

phase,liquid,ice,mixed,drizzle,liq_driz,rain,snow
height,,,,,,,
0.16,111167,234450,122302,16677,7135,21390,64136
0.19,113623,235180,122096,15269,7365,21032,62866
0.22,118208,230978,121208,14081,7954,20517,61544
0.25,121213,221319,118389,12947,9839,19855,60688
0.28,120777,210554,116419,12458,10615,19717,59582
...,...,...,...,...,...,...,...
11.53,0,61,0,0,0,0,0
11.56,0,39,0,0,0,0,0
11.59,0,25,0,0,0,0,0


In [6]:
METRIC_NAMES = Literal["precision", "recall", "f1_score", "iou", "accuracy"]


def recall_from_matrix(
    matrix: pd.DataFrame, phase: str, truth_thresh: int = 1000
) -> float:
    """Calculates recall score for a given phase using the confusion matrix.

    matrix should be a confusion matrix where index is the ground truth and the
    columns represent the model predictions.

    recall is true positives divided by true positives plus false negatives.
    """
    tp = matrix.loc[phase][phase]
    tp_fn = matrix.loc[phase].sum().item()  # by index (ground truth)
    if tp_fn < truth_thresh:
        return np.nan
    return tp / tp_fn


def precision_from_matrix(
    matrix: pd.DataFrame, phase: str, pred_thresh: int = 1000
) -> float:
    """Calculates precision score for a given phase using the confusion matrix.

    matrix should be a confusion matrix where index is the ground truth and the
    columns represent the model predictions.

    precision is true positives divided by true positives plus false positives.
    """
    tp = matrix.loc[phase][phase]
    tp_fp = matrix[phase].sum().item()  # by column (pred)
    if tp_fp < pred_thresh:
        return np.nan
    return tp / tp_fp


def f1_score_from_matrix(
    matrix: pd.DataFrame, phase: str, truth_thresh: int = 1000, pred_thresh: int = 1000
) -> float:
    """Calculates f1 score for a given phase using the confusion matrix.

    matrix should be a confusion matrix where index is the ground truth and the
    columns represent the model predictions.

    f1 score is the harmonic mean of precision and recall and is calculated as 2pr/(p+r)
    """
    p = precision_from_matrix(matrix, phase, pred_thresh=pred_thresh)
    r = recall_from_matrix(matrix, phase, truth_thresh=truth_thresh)
    if np.isnan(p) or np.isnan(r):
        return np.nan
    return (2 * p * r) / (p + r)


def iou_from_matrix(
    matrix: pd.DataFrame, phase: str, union_thresh: int = 1000
) -> float:
    """Calculates iou score for a given phase using the confusion matrix.

    matrix should be a confusion matrix where index is the ground truth and the
    columns represent the model predictions.

    IoU is true positives divided by true positives plus false positives plus false
    negatives.
    """
    tp = matrix.loc[phase][phase]
    tp_fp_fn = matrix[phase].sum().item() + matrix.loc[phase].sum().item() - tp
    if tp_fp_fn < union_thresh:
        return np.nan
    return tp / tp_fp_fn


def accuracy_from_matrix(matrix: pd.DataFrame) -> float:
    """Calculates accuracy score using the confusion matrix.

    matrix should be a confusion matrix where index is the ground truth and the
    columns represent the model predictions.

    Accuracy is calculated as true positives (the main diagonal) divided by true
    positives plus false positives and false negatives.
    """
    tp = np.diag(matrix).sum()
    tp_fp_fn = matrix.sum().sum()
    return tp / tp_fp_fn


def apply_weights(scores: dict[str, float], weights: dict[str, float] | None = None) -> float:
    scores = {k: v for k, v in scores.items() if not np.isnan(v)}
    if not len(scores):
        return np.nan
    if weights is None:
        weights = {k: 1 / len(scores) for k in scores}
    weights = {k: weights[k] for k in scores}

    score_arr = np.array(list(scores.values()))
    weight_arr = np.array(list(weights.values()))
    weight_arr /= weight_arr.sum()  # ensure adds up to 1
    return (score_arr * weight_arr).sum()

def calculate_scores_from_matrix(
    matrix: pd.DataFrame,
    metrics: list[METRIC_NAMES] | Literal["all"] = "all",
    weights: dict[str, float] | None = None,
) -> pd.Series:
    if metrics == "all":
        metrics = ["precision", "recall", "f1_score", "iou", "accuracy"]
    score_functions: dict[METRIC_NAMES, Callable[[pd.DataFrame, str], float]] = {
        "precision": precision_from_matrix,
        "recall": recall_from_matrix,
        "f1_score": f1_score_from_matrix,
        "iou": iou_from_matrix,
    }
    score_functions = {k: v for k, v in score_functions.items() if k in metrics}
    all_scores: dict[str, dict[str, float]] = {
        metric: {phase: func(matrix, phase) for phase in matrix.columns}
        for metric, func in score_functions.items()
    }
    scores = {
        metric: apply_weights(phase_scores, weights)
        for metric, phase_scores in all_scores.items()
    }
    if "accuracy" in metrics:
        scores["accuracy"] = accuracy_from_matrix(matrix)
    return pd.Series(scores)

In [9]:
def get_macro_results(pred_df: pd.DataFrame, pred_cols: list[str]) -> pd.DataFrame:
    conf_matrices = {model: pd.crosstab(pred_df["cloud_phase"], pred_df[model]) for model in pred_cols}
    model_scores = pd.DataFrame(
        {model: calculate_scores_from_matrix(conf_matrices[model]) for model in pred_cols}
    )
    return model_scores.T.round(3)


# This is Table 2. in the manuscript 
print("NSA Results:")
get_macro_results(df, MODELS)

NSA Results:


,precision,recall,f1_score,iou,accuracy
cnn,0.890,0.894,0.891,0.811,0.957
cnn_dropout,0.869,0.681,0.750,0.622,0.884
mlp_balanced,0.756,0.898,0.808,0.695,0.846
rf_balanced,0.774,0.903,0.823,0.715,0.858


In [10]:
print("ANX Results:")
get_macro_results(anx, MODELS)

ANX Results:


,precision,recall,f1_score,iou,accuracy
cnn,0.841,0.777,0.805,0.690,0.925
cnn_dropout,0.693,0.470,0.594,0.393,0.825
mlp_balanced,0.649,0.735,0.644,0.499,0.727
rf_balanced,0.697,0.807,0.722,0.590,0.792


In [11]:
# This is Table 7. in the manuscript
def get_ablation_results(pred_df: pd.DataFrame, model_prefix: str) -> pd.DataFrame:
    col_order = [
        f"{model_prefix}",
        f"{model_prefix}_mpl",
        f"{model_prefix}_mpl_b",
        f"{model_prefix}_mpl_ldr",
        f"{model_prefix}_mwr",
        f"{model_prefix}_rad",
        f"{model_prefix}_rad_ldr",
        f"{model_prefix}_rad_mdv",
        f"{model_prefix}_rad_ref",
        f"{model_prefix}_rad_spec",
        f"{model_prefix}_sonde"
    ]
    _results = {}
    for col in col_order:
        matrix = pd.crosstab(pred_df["cloud_phase"], pred_df[col], dropna=False)
        _results[col] = {}
        for phase in PHASE_ORDERS:
            _results[col][phase] = iou_from_matrix(matrix, phase)
        _results[col]["mean"] = apply_weights(_results[col])
        _results[col]["acc"] = accuracy_from_matrix(matrix) * 100

    reorder = ["drizzle", "ice", "liq_driz", "liquid", "mixed", "rain", "snow", "mean", "acc"]

    return pd.DataFrame(_results).T.round(3)[reorder]

print("NSA CNN Instrument Ablation Results:")
get_ablation_results(df, "cnn")


NSA CNN Instrument Ablation Results:


,drizzle,ice,liq_driz,liquid,mixed,rain,snow,mean,acc
cnn,0.709,0.958,0.636,0.788,0.768,0.883,0.932,0.811,95.700
cnn_mpl,0.661,0.899,0.618,0.437,0.567,0.877,0.939,0.714,90.990
cnn_mpl_b,0.675,0.902,0.593,0.445,0.598,0.886,0.931,0.718,91.302
cnn_mpl_ldr,0.662,0.917,0.633,0.587,0.613,0.894,0.937,0.749,92.424
cnn_mwr,0.706,0.954,0.637,0.779,0.754,0.885,0.932,0.807,95.384
cnn_rad,0.204,0.797,0.040,0.151,0.201,0.000,0.000,0.199,75.630
cnn_rad_ldr,0.709,0.957,0.649,0.781,0.762,0.883,0.932,0.810,95.619
cnn_rad_mdv,0.261,0.955,0.078,0.714,0.732,0.661,0.928,0.619,93.966
cnn_rad_ref,0.357,0.846,0.365,0.224,0.434,0.753,0.000,0.426,81.111
cnn_rad_spec,0.693,0.891,0.619,0.654,0.329,0.868,0.929,0.712,90.330


In [13]:
print("NSA CNN-ICD Instrument Ablation Results:")
get_ablation_results(df, "cnn_dropout")

NSA CNN-ICD Instrument Ablation Results:


,drizzle,ice,liq_driz,liquid,mixed,rain,snow,mean,acc
cnn_dropout,0.441,0.875,0.530,0.426,0.429,0.849,0.808,0.622,88.396
cnn_dropout_mpl,0.535,0.894,0.555,0.412,0.546,0.844,0.890,0.668,90.198
cnn_dropout_mpl_b,0.467,0.877,0.553,0.362,0.463,0.850,0.860,0.633,88.721
cnn_dropout_mpl_ldr,0.469,0.877,0.508,0.407,0.448,0.841,0.819,0.624,88.584
cnn_dropout_mwr,0.436,0.876,0.533,0.438,0.440,0.850,0.802,0.625,88.508
cnn_dropout_rad,0.180,0.800,0.001,0.103,0.244,0.003,0.204,0.219,76.822
cnn_dropout_rad_ldr,0.432,0.869,0.525,0.388,0.411,0.849,0.799,0.611,87.855
cnn_dropout_rad_mdv,0.347,0.891,0.374,0.488,0.467,0.694,0.836,0.585,89.207
cnn_dropout_rad_ref,0.374,0.870,0.445,0.450,0.500,0.770,0.109,0.502,84.274
cnn_dropout_rad_spec,0.459,0.879,0.470,0.600,0.316,0.802,0.873,0.629,88.884


In [14]:
print("NSA MLP Instrument Ablation Results:")
get_ablation_results(df, "mlp_balanced")

NSA MLP Instrument Ablation Results:


,drizzle,ice,liq_driz,liquid,mixed,rain,snow,mean,acc
mlp_balanced,0.677,0.810,0.571,0.499,0.489,0.928,0.893,0.695,84.645
mlp_balanced_mpl,0.658,0.834,0.509,0.490,0.511,0.925,0.877,0.686,86.101
mlp_balanced_mpl_b,0.660,0.812,0.502,0.482,0.482,0.927,0.891,0.680,84.596
mlp_balanced_mpl_ldr,0.629,0.792,0.490,0.437,0.483,0.921,0.877,0.661,83.066
mlp_balanced_mwr,0.548,0.774,0.689,0.441,0.461,0.747,0.892,0.650,81.783
mlp_balanced_rad,0.194,0.770,0.041,0.000,0.216,0.000,0.000,0.174,71.673
mlp_balanced_rad_ldr,0.660,0.778,0.546,0.452,0.459,0.926,0.899,0.674,82.320
mlp_balanced_rad_mdv,0.318,0.802,0.116,0.462,0.454,0.754,0.893,0.543,82.691
mlp_balanced_rad_ref,0.290,0.771,0.259,0.047,0.296,0.745,0.000,0.344,71.179
mlp_balanced_rad_spec,0.661,0.829,0.566,0.472,0.327,0.833,0.887,0.653,85.196


In [15]:
print("NSA RF Instrument Ablation Results:")
get_ablation_results(df, "rf_balanced")

NSA RF Instrument Ablation Results:


,drizzle,ice,liq_driz,liquid,mixed,rain,snow,mean,acc
rf_balanced,0.714,0.824,0.603,0.510,0.512,0.940,0.901,0.715,85.767
rf_balanced_mpl,0.688,0.840,0.515,0.487,0.529,0.938,0.899,0.699,86.614
rf_balanced_mpl_b,0.700,0.834,0.524,0.496,0.517,0.939,0.900,0.701,86.262
rf_balanced_mpl_ldr,0.698,0.825,0.559,0.485,0.527,0.939,0.900,0.705,85.786
rf_balanced_mwr,0.715,0.786,0.691,0.464,0.471,0.917,0.902,0.706,83.104
rf_balanced_rad,0.204,0.772,0.026,0.000,0.204,0.000,0.000,0.172,72.793
rf_balanced_rad_ldr,0.708,0.815,0.579,0.496,0.501,0.939,0.905,0.706,85.087
rf_balanced_rad_mdv,0.301,0.827,0.081,0.473,0.489,0.751,0.901,0.546,84.433
rf_balanced_rad_ref,0.284,0.790,0.260,0.030,0.311,0.766,0.000,0.348,72.694
rf_balanced_rad_spec,0.726,0.836,0.641,0.483,0.321,0.919,0.895,0.689,85.983


In [33]:
def load_counts(filepath: str) -> pd.DataFrame:
    counts = pd.read_parquet("./data/nsa_phase_counts.parquet").loc[:, "cloud_phase", :]
    counts_pivot = counts.reset_index().pivot(index="height", columns="phase", values="count")
    return counts_pivot.fillna(0).astype(int).reset_index()

nsa_counts = load_counts("./data/nsa_phase_counts.parquet")
nsa_counts

phase,height,drizzle,ice,liq_driz,liquid,mixed,rain,snow
0,0.16,16677,234450,7135,111167,122302,21390,64136
1,0.19,15269,235180,7365,113623,122096,21032,62866
2,0.22,14081,230978,7954,118208,121208,20517,61544
3,0.25,12947,221319,9839,121213,118389,19855,60688
4,0.28,12458,210554,10615,120777,116419,19717,59582
...,...,...,...,...,...,...,...,...
379,11.53,0,61,0,0,0,0,0
380,11.56,0,39,0,0,0,0,0
381,11.59,0,25,0,0,0,0,0
382,11.62,0,12,0,0,0,0,0


In [18]:
model_height_matrices: dict[str, dict[float, pd.DataFrame]] = {
    model: {
        np.round(height, 2): pd.crosstab(group["cloud_phase"], group[model], dropna=False)
        for height, group in tqdm(df.groupby("height_bin", axis="index"))
    }
    for model in tqdm(MODELS)
}  # type: ignore

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

In [34]:
_height_scores = {
    model: pd.DataFrame({
        height: calculate_scores_from_matrix(matrix)
        for height, matrix in height_matrices.items()
    }).T
    for model, height_matrices in model_height_matrices.items()
}
height_scores = pd.concat(_height_scores)
# height_scores.index.set_names(["model", "height"], inplace=True)
height_scores = height_scores.reset_index(names=["model", "height"])
height_scores

,model,height,precision,recall,f1_score,iou,accuracy
0,cnn,0.5,0.874750,0.886316,0.879181,0.791801,0.910245
1,cnn,1.0,0.902729,0.902734,0.902228,0.826916,0.926962
2,cnn,1.5,0.904928,0.904477,0.904421,0.829929,0.934497
3,cnn,2.0,0.875863,0.891785,0.882833,0.796548,0.939756
4,cnn,2.5,0.864866,0.881859,0.872053,0.781322,0.948078
...,...,...,...,...,...,...,...
87,rf,9.5,0.999249,0.999090,0.999169,0.998340,0.998336
88,rf,10.0,0.999498,1.000000,0.999749,0.999498,0.999498
89,rf,10.5,0.999313,1.000000,0.999656,0.999313,0.999313
90,rf,11.0,0.999725,1.000000,0.999863,0.999725,0.999725


In [46]:
from typing import Any

from matplotlib.colors import to_hex

def rgb_to_hex(rgb):
    """Function to convert the sns colormap/values to hex for plotly"""
    return to_hex(rgb)

def create_figure(
    size: tuple[int, int] = (1200, 800),
    legend_loc: tuple[float, float] = (0.075, 1.0),
    xaxis: dict[str, Any] | None = None,
    xaxis2: dict[str, Any] | None = None,
    yaxis: dict[str, Any] | None = None,
    yaxis2: dict[str, Any] | None = None,
    legend: dict[str, Any] | None = None,
    **kwargs,
) -> go.Figure:
    """Create a plotly.graph_objects.Figure with some preferred settings.

    Args:
        size (tuple[int, int], optional): Width, height tuple. Defaults to (1200, 800).
        legend_loc (tuple[float, float], optional): Legend location. Defaults to (0.075, 1.0).
        xaxis (dict[str, Any] | None, optional): Optional extra keyword arguments for go.Layout(xaxis=...). Defaults to None.
        xaxis2 (dict[str, Any] | None, optional): Optional extra keyword arguments for go.Layout(xaxis2=...). Defaults to None.
        yaxis (dict[str, Any] | None, optional): Optional extra keyword arguments for go.Layout(yaxis=...). Defaults to None.
        yaxis2 (dict[str, Any] | None, optional): Optional extra keyword arguments for go.Layout(yaxis2=...). Defaults to None.
        legend (dict[str, Any] | None, optional): Optional extra keyword arguments for go.Layout(legend=...). Defaults to None.

    Returns:
        plotly.graph_objects.Figure: The plotly figure with specified + preferred layout settings.
    """
    xaxis = xaxis or {}
    xaxis2 = xaxis2 or {}
    yaxis = yaxis or {}
    yaxis2 = yaxis2 or {}
    legend = legend or {}
    layout_settings = dict(
        width=size[0],
        height=size[1],
        barmode="stack",
        bargap=0,
        bargroupgap=0,
        font=dict(family="serif", size=36, color="black"),
        xaxis={
            **dict(
                showline=True,
                linewidth=2,
                linecolor="black",
                ticks="outside",
                tickwidth=2,
                tickfont_size=22,
                mirror=True,
            ),
            **xaxis,
        },
        xaxis2={
            **dict(
                showline=True,
                linewidth=2,
                linecolor="black",
                ticks="outside",
                tickwidth=2,
                tickfont_size=22,
            ),
            **xaxis2,
        },
        yaxis={
            **dict(
                side="left",
                showgrid=False,
                showline=True,
                linewidth=2,
                linecolor="black",
                ticks="outside",
                tickwidth=2,
                tickfont_size=22,
                mirror=True,
            ),
            **yaxis,
        },
        yaxis2={
            **dict(
                overlaying="y",
                side="right",
                showgrid=False,
                showline=True,
                linewidth=2,
                linecolor="black",
                ticks="outside",
                tickfont_size=22,
                tickwidth=2,
            ),
            **yaxis2,
        },
        plot_bgcolor="white",
        legend={
            **dict(
                x=legend_loc[0],
                y=legend_loc[1],
                bgcolor="rgba(0,0,0,0.1)",
                orientation="h",
                traceorder="normal",
                font_size=28,
            ),
            **legend,
        },
        margin=dict(t=0, b=0, l=0, r=10)
    )
    layout_settings.update(**kwargs)
    fig = go.Figure(layout=go.Layout(**layout_settings))
    return fig

In [60]:
def add_phase_height_counts(fig: go.Figure, counts_df: pd.DataFrame) -> None:
    count_traces = []
    for phase in PHASE_ORDERS:
        color = cblind_cmap[phase]
        if isinstance(color, tuple):
            color = rgb_to_hex(color)
        count_traces.append(
            go.Bar(
                x=counts_df[phase],
                y=counts_df["height"],
                orientation="h",
                marker_line_width=0,
                marker_color=color,
                name=phase,
                opacity=0.8,
                xaxis="x2",
            )
        )
    fig.add_traces(count_traces)

def add_height_scores(fig: go.Figure, scores_df: pd.DataFrame) -> None:
    metric_traces = []
    models_to_plot = {
        "cnn": ("CNN", "green"),
        "mlp": ("MLP", "blue"),
        "rf": ("RF", "darkorange"),
    }
    metrics_to_plot = {
        "iou": ("IoU", "dot"),
        "f1_score": ("F1", "solid"),
    }
    for metric, (metric_label, line_style) in metrics_to_plot.items():
        for model, (model_label, color) in models_to_plot.items():
            model_scores = scores_df[scores_df["model"] == model]
            metric_traces.append(
                go.Scatter(
                    x=model_scores[metric],
                    y=model_scores["height"] - 0.5,
                    line=dict(
                        color=color,
                        width=6,
                        dash=line_style,
                    ),
                    name=f"{model_label} {metric_label}",
                )
            )
    fig.add_traces(metric_traces)


def plot_height_scores(scores_df: pd.DataFrame, counts_df: pd.DataFrame, filepath: str | None = None) -> go.Figure:
    fig = create_figure(
        size=(900, 1200),
        legend_loc=(0.5, -0.13),
        xaxis=dict(title="Model Score", range=[0.0, 1.01], overlaying='x2', side="top"),
        xaxis2=dict(title="Phase Count", side="bottom"),
        yaxis=dict(title="Height (km)", range=[0.15, 10]),
        legend=dict(bgcolor=None, xanchor="center"),
    )
    add_phase_height_counts(fig, counts_df)
    add_height_scores(fig, scores_df)
    if filepath is not None:
        fig.write_image(filepath, scale=5)
    return fig

fig = plot_height_scores(height_scores, nsa_counts, filepath="figures/height_scores.png")
fig.show()

In [24]:
import xarray as xr

In [ ]:
ds = xr.open_dataset("../processing/data/nsathermocldphaseC1.c1.20210905.000000.nc")
model_prefix = {
    "cnn_20240429_213223": "cnn",
    "mlp": "mlp",
    "rf": "rf",
    "cnn_20240501_090456": "cnn_icd",
}

# cnn_icd_vars = [c for c in list(ds) if "cnn_20240501" in c]
# [c[len("cnn_20240501_090456"):].lstrip("_") for c in cnn_icd_vars]0
ds

In [ ]:
"_mwr_b".lstrip("_")

In [ ]:
cnn_icd_vars = [c for c in list(ds) if "cnn_20240501" in c]
{c: "ccn_icd" + c[len("cnn_20240501_090456"):] for c in cnn_icd_vars}